In [ ]:
# simple_test_server.py - TUZATILGAN VERSIYA
import socket
import threading
import time
from datetime import datetime

def simple_server():
    """Oddiy test server - TUZATILGAN"""
    HOST = '127.0.0.1'
    PORT = 9999
    
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((HOST, PORT))
    server.listen(5)
    
    print(f"✅ Server {HOST}:{PORT} da ishga tushdi")
    print(f"📡 Klientlarni kutmoqda...")
    
    clients = []
    
    def handle_client(client_socket, addr):
        """Klient bilan ishlash"""
        nickname = ""
        try:
            # Birinchi xabar - foydalanuvchi nomi
            nickname = client_socket.recv(1024).decode('utf-8', errors='ignore')
            if not nickname:
                nickname = f"User_{addr[1]}"
            
            print(f"👤 {nickname} ulandi: {addr[0]}")
            
            while True:
                # Xabarni olish
                data = client_socket.recv(4096)
                if not data:
                    break
                
                # Log
                timestamp = datetime.now().strftime("%H:%M:%S")
                message = data.decode('utf-8', errors='ignore')
                print(f"[{timestamp}] {nickname}: {message[:50]}...")
                
                # Boshqa klientlarga yuborish
                for other_client, other_addr in clients:
                    if other_client != client_socket:
                        try:
                            other_client.send(data)
                        except:
                            pass
                
        except Exception as e:
            print(f"❌ Xato: {e}")
        finally:
            # Klientni olib tashlash
            for i, (sock, sock_addr) in enumerate(clients):
                if sock == client_socket:
                    clients.pop(i)
                    break
            
            try:
                client_socket.close()
            except:
                pass
            
            print(f"🔌 {nickname if nickname else 'Klient'} uzildi")
    
    try:
        while True:
            client_socket, addr = server.accept()
            print(f"📥 Yangi ulanish: {addr[0]}:{addr[1]}")
            clients.append((client_socket, addr))
            
            # Yangi thread - BU YERDA XATOLIK TUG'RILASHTIRILDI
            thread = threading.Thread(
                target=handle_client, 
                args=(client_socket, addr)
            )
            thread.daemon = True
            thread.start()
            
    except KeyboardInterrupt:
        print("\n🛑 Server to'xtatildi")
    finally:
        server.close()

if __name__ == "__main__":
    simple_server()

✅ Server 127.0.0.1:9999 da ishga tushdi
📡 Klientlarni kutmoqda...
📥 Yangi ulanish: 127.0.0.1:54181
👤 User_54181 ulandi: 127.0.0.1
🔌 User_54181 uzildi
📥 Yangi ulanish: 127.0.0.1:54182
👤 asadbek ulandi: 127.0.0.1
[07:46:55] asadbek: c2Fsb20=...
